In [ ]:
!apt-get update -qq > /dev/null
!apt-get install -y wget curl psmisc net-tools xauth git xdotool dbus-x11 python3-numpy > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1
!apt-get install -f -y > /dev/null
!apt-get remove -y tightvncserver > /dev/null 2>&1 || true
!apt-get install -y xfce4 xfce4-goodies tigervnc-standalone-server tigervnc-common > /dev/null

import os
import sys
import subprocess
import secrets
import socket
import time
import shutil
import re
import glob
import urllib.request
import psutil

os.environ['USER'] = 'root'
os.environ['HOME'] = '/root'

def dump(path, label):
    print(f"----- {label} -----")
    try:
        with open(path) as f:
            print(f.read())
    except OSError as e:
        print(f"(could not read {path}: {e})")
    print(f"----- end {label} -----")

def dump_vnc_logs():
    for p in glob.glob(os.path.expanduser('~/.vnc/*.log')):
        dump(p, p)

subprocess.run("pkill -9 -f Xvnc; pkill -9 -f vncserver; pkill -9 -f websockify; pkill -9 -f google-chrome; pkill -9 -f cloudflared", shell=True)
subprocess.run("rm -f /tmp/.X1-lock /tmp/.X11-unix/X1", shell=True)

NOVNC_DIR = os.path.expanduser('~/noVNC')
if not os.path.isdir(NOVNC_DIR):
    subprocess.run(f"git clone --depth 1 https://github.com/novnc/noVNC.git {NOVNC_DIR}", shell=True, check=True)
    subprocess.run(f"git clone --depth 1 https://github.com/novnc/websockify.git {NOVNC_DIR}/utils/websockify", shell=True, check=True)
subprocess.run(f"chmod +x {NOVNC_DIR}/utils/novnc_proxy", shell=True)

req_file = os.path.join(NOVNC_DIR, 'utils', 'websockify', 'requirements.txt')
if os.path.isfile(req_file):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)

vnc_dir = os.path.expanduser('~/.vnc')
os.makedirs(vnc_dir, exist_ok=True)

vnc_password = secrets.token_hex(4)
print("=" * 50)
print(f"VNC PASSWORD: {vnc_password}")
print("=" * 50)
proc = subprocess.run(['vncpasswd', '-f'], input=(vnc_password + '\n').encode(), capture_output=True)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr.decode() if proc.stderr else "vncpasswd failed")
with open(os.path.join(vnc_dir, 'passwd'), 'wb') as f:
    f.write(proc.stdout)
os.chmod(os.path.join(vnc_dir, 'passwd'), 0o600)

if not os.path.getsize(os.path.join(vnc_dir, 'passwd')):
    raise RuntimeError("vncpasswd produced an empty password file")

xstartup = """#!/bin/bash
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
[ -r $HOME/.Xresources ] && xrdb $HOME/.Xresources
vncconfig -iconic &
exec startxfce4
"""
with open(os.path.join(vnc_dir, 'xstartup'), 'w') as f:
    f.write(xstartup)
os.chmod(os.path.join(vnc_dir, 'xstartup'), 0o755)

subprocess.run("vncserver -kill :1 > /dev/null 2>&1 || true", shell=True)
result = subprocess.run("vncserver -geometry 1360x768 -depth 24 :1", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    dump_vnc_logs()
    raise RuntimeError(result.stderr)

def wait_for_port(host, port, timeout=30, interval=0.5):
    deadline = time.time() + timeout
    while time.time() < deadline:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        try:
            s.connect((host, port))
            s.close()
            return True
        except OSError:
            time.sleep(interval)
    return False

if not wait_for_port('localhost', 5901, timeout=30):
    dump_vnc_logs()
    raise RuntimeError("VNC server (port 5901) never came up")

def safe_procnames():
    names = set()
    for p in psutil.process_iter(['name']):
        try:
            n = p.info['name']
            if n:
                names.add(n)
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass
    return names

def wait_for_xfce(timeout=30):
    deadline = time.time() + timeout
    while time.time() < deadline:
        out = subprocess.run("DISPLAY=:1 xdotool search --class xfdesktop", shell=True, capture_output=True, text=True)
        if out.returncode == 0 and out.stdout.strip():
            return True
        if {'xfwm4', 'xfdesktop', 'xfce4-panel'} & safe_procnames():
            return True
        time.sleep(1)
    return False

if not wait_for_xfce(30):
    dump_vnc_logs()
    raise RuntimeError("XFCE session did not start (black screen likely)")

xauth_path = os.path.expanduser('~/.Xauthority')
env = os.environ.copy()
env['DISPLAY'] = ':1'
if os.path.isfile(xauth_path):
    env['XAUTHORITY'] = xauth_path
else:
    print(f"warning: {xauth_path} not found, launching Chrome without XAUTHORITY set")

chrome_data_dir = os.path.expanduser('~/chrome_data')
if os.path.exists(chrome_data_dir):
    shutil.rmtree(chrome_data_dir)
os.makedirs(chrome_data_dir, exist_ok=True)

chrome_flags = [
    "google-chrome", "--no-sandbox", "--window-size=1360,768",
    "--disable-gpu", "--disable-dev-shm-usage", f"--user-data-dir={chrome_data_dir}",
    "--start-maximized", "--disable-infobars", "--disable-features=TranslateUI",
    "--disable-notifications", "--disable-crash-reporter", "--disable-default-browser-check",
    "--no-first-run", "--disable-software-rasterizer", "--disable-background-networking",
    "--disable-sync", "--disable-extensions", "--disable-component-update",
    "--password-store=basic", "--use-mock-keychain",
]
chrome_process = subprocess.Popen(chrome_flags, env=env, preexec_fn=os.setsid)

def check_chrome_process(timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        for p in psutil.process_iter(['name', 'cmdline']):
            try:
                name = (p.info['name'] or '').lower()
                cmdline = ' '.join(p.info['cmdline'] or []).lower()
            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
                continue
            if 'chrome' in name or 'google-chrome' in cmdline:
                return True
        time.sleep(1)
    return False

if not check_chrome_process():
    raise RuntimeError("Chrome process not found after launch")

def wait_for_chrome_window(timeout=30):
    candidates = [
        "DISPLAY=:1 xdotool search --class google-chrome",
        "DISPLAY=:1 xdotool search --class Google-chrome",
        "DISPLAY=:1 xdotool search --name Chrome",
    ]
    deadline = time.time() + timeout
    while time.time() < deadline:
        for cmd in candidates:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if out.returncode == 0 and out.stdout.strip():
                return True
        time.sleep(1)
    return False

if not wait_for_chrome_window(30):
    print("Chrome process exists but no window was detected via xdotool; this check may be too strict, inspect manually via the VNC session")

subprocess.run("fuser -k 6080/tcp > /dev/null 2>&1 || true", shell=True)
novnc_log_path = "/tmp/novnc.log"
novnc_log = open(novnc_log_path, "w")
novnc_process = subprocess.Popen(
    [f"{NOVNC_DIR}/utils/novnc_proxy", "--vnc", "localhost:5901", "--listen", "6080"],
    stdout=novnc_log,
    stderr=subprocess.STDOUT
)

if not wait_for_port('localhost', 6080, timeout=30):
    dump(novnc_log_path, novnc_log_path)
    raise RuntimeError("novnc_proxy failed to start on port 6080")

if novnc_process.poll() is not None:
    dump(novnc_log_path, novnc_log_path)
    raise RuntimeError("novnc_proxy exited unexpectedly")

def check_novnc_serves_ui(timeout=15):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://127.0.0.1:6080/vnc.html", timeout=3) as resp:
                if resp.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(1)
    return False

if not check_novnc_serves_ui():
    dump(novnc_log_path, novnc_log_path)
    raise RuntimeError("novnc_proxy is listening on 6080 but /vnc.html did not return 200")

subprocess.run("wget -q -N https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", shell=True)
subprocess.run("chmod +x cloudflared-linux-amd64", shell=True)

process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:6080", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

vnc_tunnel_url = None
for line in iter(process.stdout.readline, ''):
    print(line, end="")
    match = re.search(r'https://([a-zA-Z0-9-]+\.trycloudflare\.com)', line)
    if match and not vnc_tunnel_url:
        vnc_tunnel_url = match.group(0)
        time.sleep(2)
        if process.poll() is not None:
            raise RuntimeError("Cloudflare Tunnel exited immediately after establishing URL")
        print(f"\nOpen:\n{vnc_tunnel_url}/vnc.html\n")

return_code = process.wait()
if return_code:
    raise RuntimeError(f"Cloudflare Tunnel exited with code {return_code}")